In [19]:
import pandas as pd
from pathlib import Path

In [ ]:
# Schemes: wubi-06, wubi-86, wubi-98, shuangpin-xiaohe, shuangpin-microsoft, shuangpin-abc
# mode = "wubi-06"
mode = "shuangpin-xiaohe"

code = pd.read_csv(f"{mode}/{mode}-code.csv")
code = code.drop_duplicates(subset="char", keep="first")

# Keep 28 physical columns. Additional scheme keys (Microsoft's ;) use
# otherwise-unused paging columns, without changing their actual input codes.
COLUMN_KEYS = "qazwsxedcrfvtgbyhnujmik9ol0p"
if mode == "shuangpin-microsoft":
    COLUMN_KEYS = COLUMN_KEYS.replace("9", ";")
extra_keys = sorted(set(code["key2"]) - set(COLUMN_KEYS))
spare_keys = [key for key in "90" if key in COLUMN_KEYS and key not in set(code["key2"])]
if len(extra_keys) > len(spare_keys):
    raise ValueError("The scheme needs more than 28 column keys")
for extra, spare in zip(extra_keys, spare_keys):
    COLUMN_KEYS = COLUMN_KEYS.replace(spare, extra)
assert len(COLUMN_KEYS) == len(set(COLUMN_KEYS)) == 28

In [ ]:
df_zone = pd.DataFrame([{'key1': i, 'key2': j, 'chars': ''} for i in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ!@#$%^&*()' for j in COLUMN_KEYS])
for i, row in df_zone.iterrows():
    chars = code[(code['key1'] == row['key1']) & (code['key2'] == row['key2'])]['char'].values
    df_zone.loc[i, 'chars'] = ''.join(chars)

In [ ]:
piv = df_zone.pivot_table(index='key1', columns='key2', values='chars', aggfunc=lambda x: ' '.join(x)).transpose()
piv = piv[list('GFDSA!HJKLM@TREWQ#YUIOP$NBVCX%Z^&*()')].transpose()
piv = piv[list(COLUMN_KEYS)].transpose()
pd.set_option('display.max_columns', 29)
piv = piv.transpose()
piv = piv[list(COLUMN_KEYS)]

In [23]:
next = piv.map(lambda x: '')
new_piv = piv.copy()

def move(i1, j1, i2, j2):
    next.loc[i1, j1] = ('' if i1 == i2 else i2) + ('' if j1 == j2 else j2)
    new_piv.loc[i2, j2] += new_piv.loc[i1, j1][8:]
    new_piv.loc[i1, j1] = new_piv.loc[i1, j1][:8]
    print(f"Moved cell from ({i1}, {j1}) to ({i2}, {j2})")

In [ ]:
# Step 1: for really long cells, move to empty rows or columns

empty_cells = [(row, col) for row in new_piv.index for col in new_piv.columns if len(new_piv.loc[row, col]) == 0]

for _ in range(10):
    for row in "GFDSA!HJKLM@TREWQ#YUIOP$NBVCX%Z^&*()":
        for col in COLUMN_KEYS:
            cell = new_piv.loc[row, col]
            if len(cell) >= 16:
                empty_rows = [r for r, c in empty_cells if c == col]
                if len(empty_rows) > 0:
                    # find the row with the most empty cells
                    next_choices = [(r, len([c for r2, c in empty_cells if r2 == r])) for r in empty_rows]
                    next_choices.sort(key=lambda x: x[1], reverse=True)
                    move(row, col, next_choices[0][0], col)
                    empty_cells.remove((next_choices[0][0], col))

    for row in "GFDSA!HJKLM@TREWQ#YUIOP$NBVCX%Z^&*()":
        for col in COLUMN_KEYS:
            cell = new_piv.loc[row, col]
            if len(cell) >= 16:
                empty_cols = [c for r, c in empty_cells if r == row]
                if len(empty_cols) > 0:
                    # find the column with the most empty cells
                    next_choices = [(c, len([r for r, c2 in empty_cells if c2 == c])) for c in empty_cols]
                    next_choices.sort(key=lambda x: x[1], reverse=True)
                    move(row, col, row, next_choices[0][0])
                    empty_cells.remove((row, next_choices[0][0]))

In [ ]:
# Step 2: place the overflow cell with the fewest fitting destinations first.

def allocate_small_overflows(table, hints, move_cell):
    page_size = 8
    positions = [(row, col) for row in table.index for col in table.columns]
    order = {position: index for index, position in enumerate(positions)}
    neighbors = {
        source: [target for target in positions
                 if target != source and (target[0] == source[0] or target[1] == source[1])]
        for source in positions
    }
    moves = []

    while True:
        # Rebuild all options after each move using the current table and hints.
        lengths = table.stack().map(len).to_dict()
        outgoing = {position: bool(hint) for position, hint in hints.stack().items()}
        options = {}
        for source in positions:
            overflow = lengths[source] - page_size
            if overflow <= 0 or outgoing[source]:
                continue
            targets = [target for target in neighbors[source]
                       if not outgoing[target] and lengths[target] + overflow <= page_size]
            # Exact fills first, then the fullest result, then fixed grid order.
            targets.sort(key=lambda target: (page_size - lengths[target] - overflow, order[target]))
            if targets:
                options[source] = targets

        if not options:
            # If spare capacity is not directly reachable, try one bridge page.
            # Both hops change only one key. The bridge must be a sink before
            # the move, and it finishes full, so no old hint is overwritten.
            paths = {}
            for source in positions:
                overflow = lengths[source] - page_size
                if overflow <= 0 or outgoing[source]:
                    continue
                candidates = []
                for bridge in neighbors[source]:
                    if outgoing[bridge] or lengths[bridge] > page_size:
                        continue
                    forwarded = lengths[bridge] + overflow - page_size
                    if forwarded <= 0:
                        continue
                    for target in neighbors[bridge]:
                        if target == source or outgoing[target]:
                            continue
                        if lengths[target] + forwarded <= page_size:
                            candidates.append((bridge, target, forwarded))
                candidates.sort(key=lambda path: (
                    page_size - lengths[path[1]] - path[2],
                    path[2], order[path[0]], order[path[1]]))
                if candidates:
                    paths[source] = candidates
            if not paths:
                break
            source = min(paths, key=lambda position: (len(paths[position]), order[position]))
            bridge, target, _ = paths[source][0]
            move_cell(*source, *bridge)
            moves.append((source, bridge))
            move_cell(*bridge, *target)
            moves.append((bridge, target))
            continue

        # Zero-option cells cannot move; leave them for the diagnostic below.
        source = min(options, key=lambda position: (len(options[position]), order[position]))
        target = options[source][0]
        move_cell(*source, *target)
        moves.append((source, target))

    blocked = [(position, lengths[position] - page_size)
               for position in positions if lengths[position] > page_size]
    return moves, blocked


step2_moves, step2_blocked = allocate_small_overflows(new_piv, next, move)
print(f"Step 2: {len(step2_moves)} moves; {len(step2_blocked)} oversized cells remain.")


In [ ]:
# Never export a truncated layout if an input scheme still has overflow.
assert max(new_piv.stack().map(len)) <= 8, "Resolve oversized cells before exporting"
assert len("".join(new_piv.stack())) == len(code), "Candidate count changed during allocation"

df_blocks = pd.DataFrame([(i, j, new_piv.loc[i, j], len(piv.loc[i, j]), next.loc[i, j]) for i in 'GFDSA!HJKLM@TREWQ#YUIOP$NBVCX%^&*()Z' for j in COLUMN_KEYS], columns=['key1', 'key2', 'chars', 'sepr', 'hint'])
df_blocks['original'] = df_blocks.apply(lambda x: x.chars[:x.sepr] if x.sepr > -1 else x.chars, axis=1)
df_blocks['new'] = df_blocks.apply(lambda x: x.chars[x.sepr:] if x.sepr > -1 else '', axis=1)
df_blocks['hint'] = df_blocks.hint.apply(lambda x: x if len(x) < 2 else x.capitalize())
df_blocks[['key1', 'key2', 'chars', 'original', 'new', 'hint']].to_csv(f'{mode}/{mode}.csv', index=False)

In [27]:
strs = []
for i in 'GFDSA!HJKLM@TREWQ#YUIOP$NBVCX%^&*()Z':
    for j in range(8):
        strs.append(df_blocks[df_blocks.key1 == i]['chars'].apply(lambda x: x[j] if len(x) > j else '　').sum())

with open(f'{mode}/{mode}.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(strs))